# Linked UMAP brushing

Run the cells in order. **Shift-drag a rectangle in the left UMAP** to select cells. Both panels highlight those cells; the selected-cell count and metadata below update **after a brief pause**. Drag to pan and scroll to zoom. Double-click the left panel to clear the brush.

An empty brush shows all cells at their original opacity and reports **0 selected cells**. The right panel displays the same selection and does not start a brush.

This example uses original synthetic counts. **Scanpy computes the embedding; GenomeSpy only plots the supplied coordinates.** It uses the bundled GenomeSpy runtime with no Core patches and needs no Matplotlib imports or configuration.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

# Original synthetic counts with three groups and different marker expression.
rng = np.random.default_rng(0)
group_codes = np.repeat(np.arange(3), 40)
counts = rng.poisson(1, size=(120, 20)).astype(np.float32)
counts[np.arange(120), group_codes] += 8
adata = ad.AnnData(
    sparse.csr_matrix(counts),
    obs=pd.DataFrame(
        {
            "cell_type": pd.Categorical(
                np.array(["A", "B", "C"])[group_codes], categories=["C", "B", "A"]
            )
        },
        index=[f"cell-{i}" for i in range(120)],
    ),
    var=pd.DataFrame(index=["MS4A1"] + [f"gene_{i}" for i in range(1, 20)]),
)
# Explicit observation IDs survive row reordering and AnnData slicing.
adata.obs["cell_id"] = [f"demo-cell-{i:03d}" for i in range(adata.n_obs)]

## Compute UMAP with Scanpy

For your own data, keep your existing analysis workflow. Skip these analysis calls when your AnnData already has `adata.obsm["X_umap"]`. The plotting call never recomputes neighbors or an embedding.

In [ ]:
import scanpy as sc

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.pca(adata, n_comps=10, random_state=0)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=10, random_state=0)
sc.tl.umap(adata, random_state=0)

## Link the two color views

One native x/y interval brush controls both panels. They use the same UMAP dimensions and keep their own colors, guides, and drawing order. Brushing dims unselected fills and outlines to 20% of their original opacity.

The coordinate and ID copies below describe the plotted snapshot. Python uses the brush's domain bounds to find those IDs, independent of panel drawing order. This copies only two coordinate columns and the IDs, not the expression matrix.

In [ ]:
import genome_spy as gs
import genome_spy.scanpy as gsc

brush = gs.selection_interval("selected_cells", encodings=["x", "y"])
settled = gs.param("settled_cells", expr=brush.name, debounce=150)
chart = gsc.pl.umap(
    adata,
    color=["cell_type", "MS4A1"],
    obs_id="cell_id",
    selection=brush,
    zoom=True,
    size=60,
    width=280,
    height=300,
    ncols=2,
    show=False,
).add_params(settled)
plotted_coordinates = adata.obsm["X_umap"][:, :2].copy()
plotted_ids = adata.obs["cell_id"].astype(str).to_numpy(copy=True)

## Count selected cells and receive their IDs in Python

Run the next cell and wait for **Connected** before brushing. The chart highlights locally while you drag. A native debounced parameter sends only brush bounds after 150 ms without changes. A vectorized Python membership test updates `selected_ids` and `selected_obs`.

Bounds include the lower edge and exclude the upper edge, matching native interval selection. Cells sharing coordinates remain distinct observations. The table shows at most ten rows. Rerunning this display cell replaces its previous widget and connection.

In [ ]:
import asyncio
import html
import ipywidgets as widgets
from IPython.display import display

if previous_task := globals().get("selection_connection_task"):
    previous_task.cancel()
if previous_widget := globals().get("widget"):
    previous_widget.close()

widget = chart.widget(inline=True, controls=False)
status = widgets.HTML("Connecting to the UMAP…")
table = widgets.HTML()
selected_ids = []
selected_obs = adata.obs.iloc[:0].copy()
latest_selection = None
selected_brush = None
stop_selection = None


def show_selection(snapshot):
    global latest_selection, selected_ids, selected_obs
    latest_selection = snapshot
    x_range = snapshot["intervals"].get("x")
    y_range = snapshot["intervals"].get("y")
    mask = np.zeros(len(plotted_ids), dtype=bool)
    if x_range is not None and y_range is not None:
        x, y = plotted_coordinates.T
        mask = (
            (x >= x_range[0]) & (x < x_range[1]) & (y >= y_range[0]) & (y < y_range[1])
        )
    selected_ids = plotted_ids[mask].tolist()
    # Match explicit IDs, rather than treating render order as AnnData row order.
    selected_obs = adata.obs.loc[adata.obs["cell_id"].isin(selected_ids)].copy()
    status.value = f"<b>Connected — {len(selected_ids)} cells selected.</b>"
    table.value = selected_obs.head(10).to_html(escape=True) if selected_ids else ""


async def connect_selection():
    global api, selected_brush, stop_selection
    try:
        async with asyncio.timeout(30):
            api = await widget.get_embed_api()
            owner = await api.views.get(
                {"scope": [], "view": f"umap-brush-{brush.name}"}
            )
            selected_brush = await owner.params.get_selection(brush.name)
            updates = await api.params.get(settled.name)
            stop_selection = await updates.subscribe(show_selection)
            show_selection(await selected_brush.get_value())
    except Exception as error:
        status.value = html.escape(
            f"Connection failed: {type(error).__name__}: {error}"
        )


display(widget, status, table)
# Let the cell finish so the notebook can process browser replies.
selection_connection_task = asyncio.create_task(connect_selection())

## Inspect the selected observations

Brush a region above, then run this cell. These variables already live in the Python kernel; no browser call is needed. The original AnnData is unchanged.

In [ ]:
display({"selected_count": len(selected_ids), "selected_ids": selected_ids})
selected_obs

## Get the selected cells as AnnData

Run this cell after brushing to make an independent subset for your next Scanpy analysis. It copies the selected observations and their expression data only when you request it. Rerun after changing the brush to refresh `selected_adata`. With an empty brush the subset contains zero cells.

Keep the unique `cell_id` column when reordering or slicing your own AnnData. Editing coordinates after plotting does not change the plotted snapshot.

In [ ]:
selected_adata = adata[adata.obs["cell_id"].isin(selected_ids)].copy()
selected_adata

## Clear, stop listening, or export

Clear from Python by scheduling the call on the notebook event loop:

```python
clear_task = asyncio.create_task(selected_brush.clear())
```

The count returns to zero and the original appearance returns. To stop Python updates while keeping the interactive chart:

```python
stop_task = asyncio.create_task(stop_selection())
```

For complete cleanup, release the subscription and embed, then close the widget:

```python
async def close_selection():
    await stop_selection()
    await api.finalize()
    widget.close()

close_task = asyncio.create_task(close_selection())
```

Export the chart with `chart.save("linked_umap.html", inline=True)`. Brushing and linked highlighting work in that HTML without Python. The Python count, metadata table, and AnnData subset require the live notebook kernel.

This example supports rectangular brushing from the first panel. Shared additive point selection is separate work and is not enabled here.